# Membership EDA: 가격과 동시시청자수의 관계

## 목적
이 노트북은 `amount`, `product_cd`, `billing_method`, `payment_device`, `concurrent_streams`의 관계를 점검하기 위한 탐색 노트북이다.

확인할 질문은 다음과 같다.

1. 가격(`amount`)만으로 동시시청자수(`concurrent_streams`)를 설명할 수 있는가
2. `product_cd`를 함께 보면 관계가 더 명확해지는가
3. `concurrent_streams` 결측은 무작위 결측인가, 아니면 특정 상품에 집중된 구조적 결측인가


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("default")
sns.set_theme(style="whitegrid")

membership = pd.read_excel("../Dataset/Membership.xlsx")

analysis_cols = [
    "product_cd",
    "amount",
    "billing_method",
    "payment_device",
    "is_user_verified",
    "concurrent_streams",
]


def unique_sorted(series):
    return sorted(series.dropna().unique().tolist())


def unique_int_sorted(series):
    return sorted([int(x) for x in series.dropna().unique().tolist()])


membership[analysis_cols].head()


## 1. 기본 점검
먼저 분석에 사용할 핵심 변수만 확인한다.

`amount`와 `concurrent_streams`만 보면 관계가 단순해 보일 수 있으므로, 이후에는 `product_cd`와 결제 관련 변수까지 같이 본다.


In [ ]:
basic_summary = pd.DataFrame({
    "rows": [len(membership)],
    "product_cd_nunique": [membership["product_cd"].nunique()],
    "amount_nunique": [membership["amount"].nunique()],
    "billing_method_nunique": [membership["billing_method"].nunique()],
    "payment_device_nunique": [membership["payment_device"].nunique()],
    "concurrent_streams_missing": [membership["concurrent_streams"].isna().sum()],
})

display(basic_summary)
display(membership[analysis_cols].sample(10, random_state=42).sort_values(["product_cd", "amount"]))


## 2. 가격만 보면 관계가 깔끔한가
먼저 `amount` 기준으로 어떤 `payment_device`, `product_cd`, `concurrent_streams`가 묶이는지 확인한다.

이 단계의 핵심은 가격이 단독으로 동시시청자수를 결정하는지 보는 것이다.


In [ ]:
amount_summary = (
    membership.groupby("amount")
    .agg(
        rows=("amount", "size"),
        product_count=("product_cd", "nunique"),
        product_list=("product_cd", unique_sorted),
        payment_device_list=("payment_device", unique_sorted),
        billing_method_list=("billing_method", unique_sorted),
        concurrent_list=("concurrent_streams", unique_int_sorted),
        concurrent_nunique=("concurrent_streams", lambda s: s.dropna().nunique()),
        missing_count=("concurrent_streams", lambda s: s.isna().sum()),
    )
    .reset_index()
    .sort_values("amount")
)

ambiguous_amounts = amount_summary[amount_summary["concurrent_nunique"] > 1].copy()

display(
    amount_summary[
        [
            "amount",
            "rows",
            "product_count",
            "payment_device_list",
            "concurrent_list",
            "concurrent_nunique",
            "missing_count",
        ]
    ]
)

display(ambiguous_amounts[["amount", "product_count", "product_list", "concurrent_list"]])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

amount_pair_df = (
    membership.dropna(subset=["concurrent_streams"])
    .groupby(["amount", "concurrent_streams"])
    .size()
    .reset_index(name="rows")
)

sns.scatterplot(
    data=amount_pair_df,
    x="amount",
    y="concurrent_streams",
    size="rows",
    sizes=(50, 500),
    hue="concurrent_streams",
    palette="viridis",
    ax=axes[0],
    legend=False,
)
axes[0].set_title("Observed amount - concurrent_streams pairs")
axes[0].set_ylabel("concurrent_streams")
axes[0].set_yticks([1, 2, 3, 4])

sns.barplot(
    data=ambiguous_amounts,
    x="amount",
    y="concurrent_nunique",
    color="steelblue",
    ax=axes[1],
)
axes[1].set_title("Amounts linked to multiple concurrent_streams")
axes[1].set_ylabel("number of unique concurrent_streams")
axes[1].tick_params(axis="x", labelrotation=45)
axes[1].bar_label(axes[1].containers[0], padding=3, fontsize=9)

plt.tight_layout()
plt.show()


## 3. `product_cd`를 함께 보면 관계가 훨씬 선명해진다
다음 단계에서는 가격보다 상품 코드가 더 직접적으로 `concurrent_streams`를 설명하는지 확인한다.

만약 대부분의 `product_cd`가 하나의 `concurrent_streams`에만 연결된다면, 동시시청자수는 가격 속성이라기보다 상품 속성에 가깝다고 볼 수 있다.


In [ ]:
product_summary = (
    membership.groupby("product_cd")
    .agg(
        rows=("product_cd", "size"),
        amount_nunique=("amount", "nunique"),
        amount_list=("amount", unique_sorted),
        concurrent_nunique=("concurrent_streams", lambda s: s.dropna().nunique()),
        concurrent_list=("concurrent_streams", unique_int_sorted),
        missing_count=("concurrent_streams", lambda s: s.isna().sum()),
        missing_rate=("concurrent_streams", lambda s: round(s.isna().mean() * 100, 2)),
    )
    .reset_index()
    .sort_values(["rows", "product_cd"], ascending=[False, True])
)

product_rule_summary = pd.DataFrame({
    "total_products": [product_summary["product_cd"].nunique()],
    "products_with_one_stream": [(product_summary["concurrent_nunique"] == 1).sum()],
    "products_with_only_missing": [(product_summary["concurrent_nunique"] == 0).sum()],
    "products_with_multiple_streams": [(product_summary["concurrent_nunique"] > 1).sum()],
})

display(product_rule_summary)
display(product_summary.head(15))


In [ ]:
top_products = product_summary.head(12)["product_cd"].tolist()

product_stream_ct = pd.crosstab(membership["product_cd"], membership["concurrent_streams"]).reindex(top_products).fillna(0)
product_stream_ct.columns = [int(col) for col in product_stream_ct.columns]

plt.figure(figsize=(10, 6))
sns.heatmap(product_stream_ct, annot=True, fmt=".0f", cmap="Blues")
plt.title("Top products by row count: concurrent_streams distribution")
plt.xlabel("concurrent_streams")
plt.ylabel("product_cd")
plt.tight_layout()
plt.show()


## 4. `concurrent_streams` 결측은 무작위인가
이제 결측 집단만 따로 떼어 본다.

결측이 일부 상품에 몰려 있다면 단순한 누락이 아니라 상품 설계, 적재 규칙, 외부 결제 경로 문제일 가능성이 높다.


In [ ]:
only_conc_na = membership[membership["concurrent_streams"].isna()].copy()

missing_summary = (
    only_conc_na.groupby("product_cd")
    .agg(
        rows=("product_cd", "size"),
        amount_list=("amount", unique_sorted),
        billing_method_list=("billing_method", unique_sorted),
        payment_device_list=("payment_device", unique_sorted),
        verified_list=("is_user_verified", unique_sorted),
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(missing_summary)


In [ ]:
count_df = only_conc_na["product_cd"].value_counts().reset_index()
count_df.columns = ["product_cd", "count"]

what_844 = membership[membership["product_cd"] == "pk_844"].copy()
what_844_profile = (
    what_844[
        [
            "product_cd",
            "amount",
            "billing_method",
            "payment_device",
            "is_user_verified",
            "concurrent_streams",
        ]
    ]
    .drop_duplicates()
    .sort_values(["amount", "billing_method"])
)

verify_count_844 = what_844["is_user_verified"].fillna("NaN").value_counts().reset_index()
verify_count_844.columns = ["is_user_verified", "count"]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(data=count_df, x="product_cd", y="count", color="salmon", ax=axes[0])
axes[0].set_title("Products concentrated in concurrent_streams missing rows")
axes[0].tick_params(axis="x", labelrotation=45)
axes[0].bar_label(axes[0].containers[0], padding=3, fontsize=9)

sns.barplot(data=verify_count_844, x="is_user_verified", y="count", color="orchid", ax=axes[1])
axes[1].set_title("pk_844 verification mix")
axes[1].bar_label(axes[1].containers[0], padding=3, fontsize=9)

plt.tight_layout()
plt.show()

display(what_844_profile)


## 5. 해석
- `amount`만으로는 `concurrent_streams`가 일대일로 정해지지 않는다.
- 반면 다수의 `product_cd`는 하나의 `concurrent_streams`에만 연결된다.
- 따라서 동시시청자수는 가격 속성이라기보다 상품 속성에 더 가깝다.
- `concurrent_streams` 결측은 `pk_844` 등 소수 상품에 집중되어 있어 무작위 결측으로 보기 어렵다.
- 특히 `pk_844`는 `amount=10.99`, `billing_method=140`, `payment_device=ios`로 고정되어 있어 구조적 결측 가능성이 높다.

## 6. 후속 확인 포인트
1. `pk_844`, `pk_846`, `pk_847`, `pk_848`이 실제로 어떤 요금제 묶음인지 상품 설명서를 통해 확인한다.
2. `billing_method=140`, `payment_device=ios` 경로에서 `concurrent_streams`가 누락되는 적재 로직이 있는지 확인한다.
3. 모델링 단계에서는 행 삭제보다 `concurrent_streams_missing_flag`를 먼저 두고 영향 비교를 진행한다.
